# Training splits on the curated KG

Builds the training splits for retraining NetMedGPT on the curated KG (`edges_preprocessed.csv`: no drug_drug edges,
drugs with broad indications and misclassified drug-ADR edges removed), to test whether performance is preserved.

- **train.csv**: the edges of the original split that are also in the curated KG (all other training edges removed).
- **valid.csv / test.csv**: copied unchanged, so the retrained model is evaluated on exactly the same pairs as the
  existing checkpoints.
- Node list, node features and vocabulary stay those of the full KG (`nodes.csv`, `embeddings.pt`), so no re-indexing
  is needed; drugs removed in the curated KG simply have no training edges.

Edges are matched by (node type, node ID) of both ends and the relation (`rev_<relation>` rows of the splits are the
same KG relation), independent of direction.
Output: `data/splits_curated/<split folder>/{train,valid,test}.csv` and `summary.csv`.

In [ ]:
import os
if os.path.basename(os.getcwd()) == "curated_kg_training":
    os.chdir("../..")
import sys
sys.path.insert(0, os.getcwd())
import shutil
import pandas as pd

from netmedgpt.prepare_txgnn_splits import normalize_id

SPLITS = {"random_link_split": "random_{seed}", "zero_shot_split": "complex_disease_{seed}"}
SEEDS = [1, 2, 3, 4, 5]
CURATED_NODES = "data/KG/nodes_preprocessed.csv"
CURATED_EDGES = "data/KG/edges_preprocessed.csv"
OUT_ROOT = "data/splits_curated"

In [ ]:
# ── Curated KG edges as keys: (relation, unordered pair of (type, id)) ───────
nodes_c = pd.read_csv(CURATED_NODES, dtype={"node_id": str}, low_memory=False)
key_of = dict(zip(nodes_c["node_index"], zip(nodes_c["node_type"], nodes_c["node_id"].map(normalize_id))))
curated_drugs = set(nodes_c.loc[nodes_c["node_type"] == "drug", "node_id"])

edges_c = pd.read_csv(CURATED_EDGES, usecols=["relation", "x_index", "y_index"])
curated = set(zip(edges_c["relation"], map(frozenset, zip(edges_c["x_index"].map(key_of), edges_c["y_index"].map(key_of)))))
print(f"curated KG: {len(nodes_c)} nodes, {len(curated)} edges (relation + unordered pair), {len(curated_drugs)} drugs")


def edge_keys(df):
    rel = df["relation"].str.replace(r"^rev_", "", regex=True)
    x = list(zip(df["x_type"], df["x_id"].map(normalize_id)))
    y = list(zip(df["y_type"], df["y_id"].map(normalize_id)))
    return [(r, frozenset((a, b))) for r, a, b in zip(rel, x, y)]

In [ ]:
# ── Filter the training edges, copy val / test unchanged ─────────────────────
summary = []
for inference, folder in SPLITS.items():
    for seed in SEEDS:
        src = os.path.join("data/splits", folder.format(seed=seed))
        dst = os.path.join(OUT_ROOT, folder.format(seed=seed))
        os.makedirs(dst, exist_ok=True)

        train = pd.read_csv(os.path.join(src, "train.csv"), low_memory=False)
        keep = pd.Series([k in curated for k in edge_keys(train)], index=train.index)
        train[keep].to_csv(os.path.join(dst, "train.csv"), index=False)
        for f in ["valid.csv", "test.csv"]:
            shutil.copy(os.path.join(src, f), os.path.join(dst, f))

        # how many test drug-disease edges involve a drug that is not in the curated KG (no training edges any more)
        test = pd.read_csv(os.path.join(src, "test.csv"), low_memory=False)
        dd = test[(test["x_type"] == "drug") | (test["y_type"] == "drug")]
        drug_id = dd["x_id"].where(dd["x_type"] == "drug", dd["y_id"]).map(normalize_id)
        removed = train.loc[~keep, "relation"].str.replace(r"^rev_", "", regex=True).value_counts()
        summary.append({"inference": inference, "seed": seed, "train_rows": len(train), "train_rows_kept": int(keep.sum()),
                        "removed_drug_drug": int(removed.get("drug_drug", 0)),
                        "removed_other": int(removed.drop("drug_drug", errors="ignore").sum()),
                        "test_drug_edges": len(dd), "test_drug_edges_with_removed_drug": int((~drug_id.isin(curated_drugs)).sum())})
        print(f"{inference} seed {seed}: kept {keep.mean():.1%} of the training rows -> {dst}")
        print("  removed per relation:", removed.head(8).to_dict())

summary = pd.DataFrame(summary)
summary.to_csv(os.path.join(OUT_ROOT, "summary.csv"), index=False)
summary